# Build a Tamil Voice Agent on Pipecat with Tamil Turn Detection

This recipe builds a **real-time Tamil voice agent** with **[Pipecat](https://docs.pipecat.ai)** and **[Sarvam AI](https://docs.sarvam.ai)**, and swaps in an **end-of-turn detection model trained on Tamil**.

```
User audio -> VAD -> Tamil turn detector -> STT -> LLM -> TTS -> Audio out
                          |
                     "has the speaker finished, or are they still thinking?"
```

## Why this recipe exists

A voice agent has to decide, at every pause, whether the caller has finished speaking. Pipecat answers this with a semantic turn detector, and runs `LocalSmartTurnAnalyzerV3` **by default** — you get it without configuring anything.

The released Smart Turn v3 weights cover 23 languages. **Tamil is not one of them.** So a Tamil agent built on the defaults runs a turn detector that was never trained on Tamil prosody, and falls back to interrupting or waiting.

This recipe points the same analyzer at a Tamil checkpoint. It is a one-argument change, no new services, and no extra latency worth measuring.

| | |
|---|---|
| **Turn detection** | `santhosh-005/smart-turn-tamil` (int8 ONNX, 8.7 MB, CPU) |
| **Speech-to-text** | Sarvam **Saaras v3** (`saaras:v3`) |
| **LLM** | Sarvam `sarvam-105b` |
| **Text-to-speech** | Sarvam **Bulbul v3** (`bulbul:v3`) |
| **Transport** | [Daily](https://daily.co) or browser WebRTC |

> **Note on running this notebook:** a voice agent is a long-running process that waits for a caller to join, so it cannot be run to completion in a notebook cell. This notebook writes `agent.py` for you with `%%writefile`, then shows the terminal command to run it.

In [ ]:
%pip install -Uqq "pipecat-ai[daily,sarvam,silero]" huggingface-hub python-dotenv loguru

## Prerequisites

- Python 3.10 or higher.
- A [Sarvam AI](https://dashboard.sarvam.ai) API key.

## 1. Configure your Sarvam API key

Never commit real API keys. The cell below writes a `.env.example` template; copy it to `.env` and fill in your real key locally:

```bash
cp .env.example .env
```

In [ ]:
%%writefile .env.example
# Copy this file to `.env` and fill in your real credentials.
# Do not commit `.env` to version control.

SARVAM_API_KEY=your-sarvam-api-key

## 2. Fetch the Tamil turn detection model

The model is a public Hugging Face repo, so this needs no token. It downloads once and is cached.

Two variants ship in that repo. `tiny` is the default here because it is a drop-in size match for upstream Smart Turn v3; `base` is more accurate and larger.

| variant | file | size | accuracy | p50, 1 CPU thread |
|---|---|---|---|---|
| **tiny** | `smart-turn-tamil-tiny/smart-turn-tamil-int8-dynamic.onnx` | 8.7 MB | 83.71% | 83 ms |
| base | `smart-turn-tamil-base/smart-turn-tamil-int8-dynamic.onnx` | 21 MB | 86.13% | 143 ms |

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv(override=True)

if not os.getenv("SARVAM_API_KEY"):
    raise RuntimeError(
        "SARVAM_API_KEY is not set. Copy .env.example to .env and add your key "
        "from https://dashboard.sarvam.ai"
    )

TURN_MODEL_REPO = "santhosh-005/smart-turn-tamil"
TURN_MODEL_FILE = "smart-turn-tamil-tiny/smart-turn-tamil-int8-dynamic.onnx"

model_path = Path(hf_hub_download(TURN_MODEL_REPO, TURN_MODEL_FILE))
print(f"Turn detection model: {model_path.name}")
print(f"Size: {model_path.stat().st_size / 1e6:.1f} MB")
print(f"Cached at: {model_path.parent}")


## 3. Write the voice agent

The pipeline is Pipecat's standard voice-agent shape: transport input, STT, user context, LLM, TTS, transport output, assistant context.

The turn detector does not sit in that list. It is a **user-turn-stop strategy**, passed through `LLMUserAggregatorParams`, and it decides when the user aggregator closes a turn and lets the LLM run:

```python
UserTurnStrategies(
    stop=[TurnAnalyzerUserTurnStopStrategy(
        turn_analyzer=LocalSmartTurnAnalyzerV3(smart_turn_model_path=model_path)
    )]
)
```

Without that argument, Pipecat builds the same strategy with the default Smart Turn v3 weights.

`%%writefile` saves this cell to `agent.py` in the current directory.

In [ ]:
%%writefile agent.py
"""Tamil voice agent on Pipecat, with a Tamil end-of-turn detection model."""
from __future__ import annotations

import os

from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from loguru import logger
from pipecat.audio.turn.smart_turn.local_smart_turn_v3 import LocalSmartTurnAnalyzerV3
from pipecat.audio.vad.silero import SileroVADAnalyzer
from pipecat.frames.frames import LLMRunFrame
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.runner import PipelineRunner
from pipecat.pipeline.task import PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
    LLMUserAggregatorParams,
)
from pipecat.runner.types import RunnerArguments
from pipecat.runner.utils import create_transport
from pipecat.services.sarvam.llm import SarvamLLMService
from pipecat.services.sarvam.stt import SarvamSTTService
from pipecat.services.sarvam.tts import SarvamTTSService
from pipecat.transcriptions.language import Language
from pipecat.transports.base_transport import TransportParams
from pipecat.transports.daily.transport import DailyParams
from pipecat.turns.user_stop import TurnAnalyzerUserTurnStopStrategy
from pipecat.turns.user_turn_strategies import UserTurnStrategies

load_dotenv(override=True)

TURN_MODEL_REPO = "santhosh-005/smart-turn-tamil"
TURN_MODEL_FILE = "smart-turn-tamil-tiny/smart-turn-tamil-int8-dynamic.onnx"

SYSTEM_PROMPT = (
    "\u0BA8\u0BC0\u0B99\u0BCD\u0B95\u0BB3\u0BCD \u0B92\u0BB0\u0BC1 \u0B89\u0BA4\u0BB5\u0BBF\u0BAF\u0BBE\u0BB3\u0BB0\u0BCD. "
    "\u0BA4\u0BAE\u0BBF\u0BB4\u0BBF\u0BB2\u0BCD \u0BAE\u0B9F\u0BCD\u0B9F\u0BC1\u0BAE\u0BCD, \u0B9A\u0BC1\u0BB0\u0BC1\u0B95\u0BCD\u0B95\u0BAE\u0BBE\u0B95 \u0BAA\u0BA4\u0BBF\u0BB2\u0BB3\u0BBF\u0B95\u0BCD\u0B95\u0BB5\u0BC1\u0BAE\u0BCD. "
    "(You are a helpful assistant. Reply only in Tamil, in at most two sentences.)"
)


def tamil_turn_strategies() -> UserTurnStrategies:
    """Point Pipecat's own turn analyzer at the Tamil checkpoint.

    Pipecat already runs LocalSmartTurnAnalyzerV3 as its default user-turn-stop
    strategy. The released Smart Turn v3 weights cover 23 languages and Tamil is
    not one of them, so the only change needed is the model path.
    """
    model_path = hf_hub_download(TURN_MODEL_REPO, TURN_MODEL_FILE)
    logger.info(f"Tamil turn detection model: {model_path}")

    # Pass the .onnx file, not its directory.
    analyzer = LocalSmartTurnAnalyzerV3(smart_turn_model_path=model_path)
    return UserTurnStrategies(
        stop=[TurnAnalyzerUserTurnStopStrategy(turn_analyzer=analyzer)]
    )


async def bot(runner_args: RunnerArguments) -> None:
    """Main bot entry point."""
    transport = await create_transport(
        runner_args,
        {
            "daily": lambda: DailyParams(audio_in_enabled=True, audio_out_enabled=True),
            "webrtc": lambda: TransportParams(
                audio_in_enabled=True, audio_out_enabled=True
            ),
        },
    )

    api_key = os.getenv("SARVAM_API_KEY")

    stt = SarvamSTTService(
        api_key=api_key,
        mode="transcribe",
        settings=SarvamSTTService.Settings(model="saaras:v3", language=Language.TA_IN),
    )
    tts = SarvamTTSService(
        api_key=api_key,
        settings=SarvamTTSService.Settings(
            model="bulbul:v3", language=Language.TA_IN, voice="shubh"
        ),
    )
    llm = SarvamLLMService(
        api_key=api_key,
        settings=SarvamLLMService.Settings(model="sarvam-105b"),
    )

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    context = LLMContext(messages)

    # The only difference from the base Pipecat recipe: user_params.
    context_aggregator = LLMContextAggregatorPair(
        context,
        user_params=LLMUserAggregatorParams(
            vad_analyzer=SileroVADAnalyzer(),
            user_turn_strategies=tamil_turn_strategies(),
        ),
    )

    pipeline = Pipeline(
        [
            transport.input(),
            stt,
            context_aggregator.user(),
            llm,
            tts,
            transport.output(),
            context_aggregator.assistant(),
        ]
    )

    task = PipelineTask(pipeline)

    @transport.event_handler("on_client_connected")
    async def on_client_connected(transport, client) -> None:
        logger.info("Client connected")
        messages.append(
            {"role": "system", "content": "Greet the caller in Tamil in one sentence."}
        )
        await task.queue_frames([LLMRunFrame()])

    @transport.event_handler("on_client_disconnected")
    async def on_client_disconnected(transport, client) -> None:
        logger.info("Client disconnected")
        await task.cancel()

    runner = PipelineRunner(handle_sigint=runner_args.handle_sigint)
    await runner.run(task)


if __name__ == "__main__":
    from pipecat.runner.run import main

    main()


## 4. Run and test your agent

From a terminal in the same directory as `agent.py` and `.env`:

```bash
python agent.py
```

The agent creates a Daily room and prints a URL. Open it, speak Tamil, and pause mid-sentence: the agent should wait for you rather than cutting in.

## What changed from the base Pipecat recipe

Compared with `integrations/build_voice_agent_with_pipecat.ipynb`, three things:

**1. The turn detector is Tamil.** One extra argument on the context aggregator.

```python
context_aggregator = LLMContextAggregatorPair(
    context,
    user_params=LLMUserAggregatorParams(
        vad_analyzer=SileroVADAnalyzer(),
        user_turn_strategies=tamil_turn_strategies(),
    ),
)
```

**2. STT and TTS are configured through `settings=`.** On Pipecat 1.7 the top-level `language=` and `target_language_code=` arguments are no longer read, so language has to go through the settings object:

```python
stt = SarvamSTTService(
    api_key=api_key,
    mode="transcribe",
    settings=SarvamSTTService.Settings(model="saaras:v3", language=Language.TA_IN),
)
```

**3. The system prompt is Tamil**, so the agent replies in Tamil and there are real turns to detect.

## About the turn detection model

A fine-tune of [Smart Turn v3](https://github.com/pipecat-ai/smart-turn) on Tamil telephone speech. It reads prosody from the last 8 seconds of the caller's audio and predicts completion, before a transcript exists.

| | accuracy | ROC-AUC | FP rate |
|---|---|---|---|
| always assume "finished" | 63.08% | 0.500 | - |
| Smart Turn v3.2, zero-shot on Tamil | 70.30% | 0.751 | - |
| **smart-turn-tamil, tiny int8** | **83.71%** | **0.905** | 7.94% |
| smart-turn-tamil, base int8 | 86.13% | 0.921 | 9.17% |

Scored on 4,168 held-out clips from 30 calls never used in training. A false positive here means the agent talks over the caller.

**Training data:** 18,485 labelled turn boundaries from 116 real Tamil two-party telephone conversations, derived from [SPRING-INX Tamil R1](https://arxiv.org/abs/2310.14654) (SPRING Lab, IIT Madras, CC BY 4.0). Real narrowband telephone audio, not TTS.

- Model: https://huggingface.co/santhosh-005/smart-turn-tamil
- Dataset: https://huggingface.co/datasets/santhosh-005/tamil-eot
- Method, benchmarks and negative results: https://github.com/santhosh-005/tamil-eot

Model weights are BSD-2-Clause; the dataset is CC BY 4.0.

## Customization

### Use the more accurate model

```python
TURN_MODEL_FILE = "smart-turn-tamil-base/smart-turn-tamil-int8-dynamic.onnx"
```

2.42 accuracy points for 2.4x the file size and 1.7x the latency. Both are realtime.

### Give the detector more CPU

```python
LocalSmartTurnAnalyzerV3(smart_turn_model_path=model_path, cpu_count=4)
```

Defaults to 1 thread, which is the right default on a server carrying concurrent calls.

### Let the detector decide without waiting for a transcript

```python
TurnAnalyzerUserTurnStopStrategy(turn_analyzer=analyzer, wait_for_transcript=False)
```

Lower latency, at the cost of committing the turn before STT confirms it.

### Keep upstream Smart Turn for other languages

Drop `user_turn_strategies` entirely and Pipecat uses the default Smart Turn v3 weights, which cover 23 languages including Hindi, Bengali and Marathi.

## Troubleshooting

- **`INVALID_PROTOBUF: Protobuf parsing failed`** - `smart_turn_model_path` is pointing at a directory. It wants the `.onnx` file itself.
- **The agent replies in English** - check that `language=Language.TA_IN` is inside `SarvamSTTService.Settings(...)` and `SarvamTTSService.Settings(...)`. Passing `language=` at the top level is silently ignored on Pipecat 1.7.
- **The agent still interrupts** - confirm `user_turn_strategies` is actually reaching the aggregator; without it Pipecat falls back to the default Smart Turn v3 weights, which have no Tamil.
- **Model not found** - the Hugging Face repo is public and needs no token. Check network access to `huggingface.co`.
- **API key errors** - `.env` must sit in the same directory as `agent.py`.

## Additional resources

- [Sarvam AI Documentation](https://docs.sarvam.ai)
- [Pipecat Documentation](https://docs.pipecat.ai)
- [Pipecat Smart Turn overview](https://docs.pipecat.ai/api-reference/server/utilities/turn-detection/smart-turn-overview)
- [Build a voice agent with Pipecat](../../integrations/build_voice_agent_with_pipecat.ipynb)

## Need help?

- Sarvam Support: developer@sarvam.ai
- Community: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)